<a href="https://colab.research.google.com/github/RushiKP14/Tensorflow/blob/main/fcc_predict_health_costs_with_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
#Check whether there is any unknown values.
dataset.isna().sum()

In [ ]:
len(dataset['children'].unique())

In [ ]:
#Splitting dataset into training and testing set

#print(len(dataset))
train_dataset = dataset.sample(frac = 0.8)
test_dataset = dataset.drop(train_dataset.index)
#print(train_dataset.shape)
#print(len(test_dataset))
dft = train_dataset.reset_index(drop=True)
dftest = test_dataset.reset_index(drop=True)
dftrain = dft.copy()
y_train = dftrain.pop('expenses')
test_labels = dftest.pop('expenses')
dftrain.head()


In [ ]:
#creating symbolic input dictionary
inputs = {}

for name, column in dftrain.items():
  dtype = column.dtype
  if dtype == object:
    dtype = tf.string
  else:
    dtype = tf.float32

  inputs[name] = tf.keras.Input(shape=(1,), name=name, dtype=dtype)

inputs

In [ ]:
#normalization of numeric inupts
numeric_inputs = {name:input for name,input in inputs.items()
                  if input.dtype==tf.float32}

x = layers.Concatenate()(list(numeric_inputs.values()))
norm = layers.Normalization()
norm.adapt(np.array(dft[numeric_inputs.keys()]))
all_numeric_inputs = norm(x)

all_numeric_inputs

In [ ]:
#Collect all the symbolic preprocessing results, to concatenate them later
preprocessed_inputs = [all_numeric_inputs]

In [ ]:
#categorical inputs
for name, input in inputs.items():
  if input.dtype == tf.float32:
    continue

  lookup = layers.StringLookup(vocabulary=np.unique(dft[name]))
  one_hot = layers.CategoryEncoding(num_tokens=lookup.vocabulary_size())

  x = lookup(input)
  x = one_hot(x)
  preprocessed_inputs.append(x)

In [ ]:
preprocessed_inputs_cat = layers.Concatenate()(preprocessed_inputs)

health_preprocessing = tf.keras.Model(inputs, preprocessed_inputs_cat)

In [ ]:
#CATEGORICAL_COLUMNS = ['sex', 'children', 'smoker', 'region']
#for feature_name in CATEGORICAL_COLUMNS:
#  dataset = pd.get_dummies(dataset, columns=[feature_name], prefix='', prefix_sep='', dtype=float)
#dataset.tail()

In [ ]:
#creating a dictionary of tensors for input
health_features_dict = {name: np.array(value)
                         for name, value in dftrain.items()}

In [ ]:
#Testing preprocessing
features_dict = {name:values[:1] for name, values in health_features_dict.items()}
health_preprocessing(features_dict)

In [ ]:
def health_model(preprocessing_head, inputs):
  body = tf.keras.Sequential([
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
  ])

  preprocessed_inputs = preprocessing_head(inputs)
  result = body(preprocessed_inputs)
  model = tf.keras.Model(inputs, result)

  model.compile(loss='mean_absolute_error',
                optimizer=tf.keras.optimizers.Adam(0.01),
                metrics=['mae', 'mse'])
  return model

model = health_model(health_preprocessing, inputs)

In [ ]:
model.fit(x=health_features_dict, y=y_train, epochs=30)

In [ ]:
test_dataset = {name: np.array(value)
                         for name, value in dftest.items()}

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
